# Module 2 · Basic RAG on the same questions

Same filing, same three questions as Module 1. This time we **chunk the filing once, embed the chunks, and for each question send only the top 5**. Watch input tokens fall from ~100K to ~3K.

```python
# once
chunks = split(read("acme_10q.txt"), size=500, overlap=50)
vectors = embed(chunks)
# every question
def answer(q):
    top = nearest(embed(q), vectors, k=5)
    prompt = context(top) + q
    return llm(prompt)
```

No agent yet: the same pipeline runs on every message, with the query as typed, and no chat history.

**Setup:** see `README.md` (`pip install -e .`, then `.env` names the models and server). Run with the *Agentic RAG demos* kernel.
The filing is synthetic. Regenerate it with `python data/make_acme_10q.py`.

In [ ]:
import os
import time
from pathlib import Path

import litellm
import numpy as np
import pandas as pd
import tiktoken

from tools.search_docs import QUERY_PREFIX # an instruction some embedding models want on queries
from tools.settings import EMBED_MODEL, LLM_KWARGS, MODEL, ROOT, price # reads .env: models, server, price
K = 5

doc = (ROOT / "data" / "acme_10q.txt").read_text() # ROOT: works from demos/
enc = tiktoken.get_encoding("o200k_base")
doc_tokens = len(enc.encode(doc))
print(f"Model: {MODEL} Embeddings: {EMBED_MODEL}")
print(f"Document: {doc_tokens:,} tokens. v0 sent all of them with every question.")

## Offline, once · 1. Chunk

Cut the filing into ~500-token chunks with ~50 tokens of overlap, so a sentence on a boundary is not lost.
We split on paragraphs (and on table rows when a table is too big), rather than at an arbitrary token, so a chunk keeps its meaning.
Each chunk also remembers which sections of the filing it spans, so we can see where retrieved text lives.

In [ ]:
def ntok(text: str) -> int:
    return len(enc.encode(text))


def is_heading(line: str) -> bool:
    return line.isupper() or line.startswith("Note ")


def split(text: str, size: int = 500, overlap: int = 50) -> list[dict]:
    pieces, section = [], ""
    for para in text.split("\n\n"):
        first = para.strip().split("\n")[0]
        if is_heading(first):
            section = first
        parts = [para] if ntok(para) <= size else para.split("\n")
        pieces += [(p, section) for p in parts]

    chunks, cur, sections = [], [], []
    for piece, section in pieces:
        if cur and ntok("\n\n".join(cur + [piece])) > size:
            chunks.append({"text": "\n\n".join(cur), "sections": sections})
            tail = enc.decode(enc.encode(chunks[-1]["text"])[-overlap:])
            cur, sections = [tail], []
        cur.append(piece)
        if section not in sections:
            sections.append(section)
    chunks.append({"text": "\n\n".join(cur), "sections": sections})
    return chunks


def label(chunk: dict) -> str:
    return " → ".join(s.split(" (")[0][:40] for s in chunk["sections"])


chunks = split(doc)
sizes = [ntok(c["text"]) for c in chunks]
print(f"{len(chunks)} chunks, {np.mean(sizes):.0f} tokens on average (max {max(sizes)})\n")
i = next(i for i, c in enumerate(chunks) if "Acme Corp reported strong growth" in c["text"])
text = chunks[i]["text"]
print(f"chunk {i} · {label(chunks[i])} · {ntok(text)} tokens\n")
print("…" + text[text.index("Overview"):][:600] + "…")

## Offline, once · 2. Embed

An embedding model turns each chunk into a vector. Texts with similar meaning land close together, even without shared words.
We normalize the vectors so cosine similarity is a dot product. This array is our whole vector index.

In [ ]:
def embed(texts: list[str]) -> np.ndarray:
    vectors = []
    for i in range(0, len(texts), 32):
        resp = litellm.embedding(model=EMBED_MODEL, input=texts[i:i + 32])
        vectors += [d["embedding"] for d in resp.data]
    v = np.array(vectors, dtype=np.float32)
    return v / np.linalg.norm(v, axis=1, keepdims=True)


start = time.perf_counter()
vectors = embed([c["text"] for c in chunks])
print(f"Index: {vectors.shape[0]} vectors × {vectors.shape[1]} dims, built in {time.perf_counter() - start:.1f}s")
try:
    embed_cost = litellm.cost_per_token(model=EMBED_MODEL, prompt_tokens=sum(sizes))[0]
    print(f"One-time embedding cost: ${embed_cost:.4f}")
except Exception: # model missing from LiteLLM's price list
    pass

## Online, per question · 3. Retrieve top-k, generate from those

Embed the question the same way, take the 5 nearest chunks, and put only those in the prompt:

`prompt = "Answer using only this context:\n" + chunk_7 + chunk_42 + … + "\nQ: What was Q3 revenue?"`

Every question is sent on its own. Module 2 keeps no chat history.

In [ ]:
PROMPT = """Answer using only this context. If the context does not contain the answer, say so.

{context}

Q: {question}"""

v0_cost = price(doc_tokens, 0)
runs = []


def nearest(question: str, k: int = K) -> list[tuple[int, float]]:
    scores = vectors @ embed([QUERY_PREFIX + question])[0]
    top = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in top]


def answer(question: str) -> None:
    top = nearest(question)
    print(f"› {question}\n\nRetrieved top-{K}:")
    for i, score in top:
        print(f" chunk {i:3} {score:.3f} {label(chunks[i])}")

    context = "\n\n---\n\n".join(f"[chunk {i}]\n{chunks[i]['text']}" for i, _ in top)
    start = time.perf_counter()
    messages = [{"role": "user", "content": PROMPT.format(context=context, question=question)}]
    stream = litellm.completion(model=MODEL, messages=messages, stream=True,
                                stream_options={"include_usage": True}, **LLM_KWARGS)
    reply = litellm.stream_chunk_builder(list(stream), messages=messages)
    latency = time.perf_counter() - start
    print("\n" + reply.choices[0].message.content)

    usage = reply.usage
    cost = price(usage.prompt_tokens, usage.completion_tokens)
    runs.append({"question": question, "prompt_tokens": usage.prompt_tokens,
                 "completion_tokens": usage.completion_tokens, "latency_s": round(latency, 2),
                 "cost_usd": cost})
    print(f"\nprompt_tokens={usage.prompt_tokens:,} (v0: ~{doc_tokens:,}) "
          f"completion_tokens={usage.completion_tokens:,} latency={latency:.2f}s "
          f"cost={'$%.4f' % cost if cost is not None else 'n/a'}"
          + (f" (v0 input alone: ${v0_cost:.2f})" if v0_cost else ""))

## The same three questions as Module 1

In [ ]:
answer("What was Q3 revenue?")

Module 1 answered the next two from chat history. Module 2 has none, so each follow-up is searched **exactly as typed**.
Look at what comes back: "How about Q2?" has no subject, and "And the margin?" does not say which quarter.

In [ ]:
answer("How about Q2?")

In [ ]:
answer("And the margin?")

## Module 1 vs Module 2

In [ ]:
df = pd.DataFrame(runs)
df["v0_prompt_tokens"] = doc_tokens
df["reduction"] = (doc_tokens / df["prompt_tokens"]).round().astype(int).astype(str) + "×"
df

## Where a fixed pipeline breaks

Module 2 fixed the cost. Now two messages that show its limits.

In [ ]:
answer("Thanks!")

It retrieved and paid for ~3K tokens of context to reply to "Thanks!". Nothing decides *whether* to search.

Now a comparison. Q3 is discussed in the MD&A; Q2 only appears in Appendix A, the previously reported second quarter.

In [ ]:
answer("Compare Q3 and Q2 revenue.")

In [ ]:
def rank_of(question: str, section_prefix: str) -> None:
    scores = vectors @ embed([question])[0]
    order = list(np.argsort(-scores))
    best = min(order.index(i) for i, c in enumerate(chunks)
               if any(s.startswith(section_prefix) for s in c["sections"]))
    print(f"Best '{section_prefix}' chunk ranked #{best + 1} of {len(chunks)} (we only send the top {K}).")


rank_of("Compare Q3 and Q2 revenue.", "APPENDIX A")

One query vector lands near the Q3 discussion, so the Q2 numbers never make the top 5. Fixing this means **planning two searches**, one per quarter.

| Message | What Module 2 does | What would fix it | Where |
|---|---|---|---|
| "Thanks, that helps!" | Retrieves anyway | Decide whether to search | Module 3 |
| "Compare Q3 and Q2" | One query, misses Q2 | Plan two searches | Module 3 |
| "And the quarter before?" | No memory of the chat | Short-term memory | Module 3 |
| "how'd we do lately" | Weak match | Grade, rewrite and retry | Module 4 |
| "error code E-4021" | Vector search misses it | Hybrid BM25 + vector | [opensearch-in-action](https://github.com/harshuljain13/opensearch-in-action) |

**Module 2 fixed the cost: ~100K → ~3K input tokens per question.** What it still cannot do is decide anything: it retrieves on every message, once, with the query as typed. Most fixes need the model to make decisions, and a model that decides, with tools, in a loop, is an agent. That is Module 3: `module_3_give_the_model_judgment.ipynb`.